**Code Breakdown of TikTok Social Spaces Database**

In [ ]:
import json
import pandas as pd
import pyarrow.parquet as pq

# For sentiment analysis
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

pf = pq.ParquetFile("metadata.parquet")

nltk.download('vader_lexicon') # Download the VADER lexicon for sentiment analysis

I imported json, pandas to load Python libraries to be utilized. Additional imports include pyarrow.parquet to create tables and clean data, and nltk was downloaded at the end of my project to perform sentiment analysis on all 4810 comment entries. To initiate the sentiment analysis I downloaded the vader_lexicon, which is a tool that analyses social media comments, specifically categorizing the comments to be either positive, negative, or neutral. 

In [ ]:
pf = pq.ParquetFile("metadata.parquet")

Loaded a large-scale database found through Hugging Face (Shofo/shofo-tiktok-general-small) and loaded only the metadata. Originally, I attempted to download all the videos from the shofo-tiktok dataset, but was unable to as my computer was unable to download the videos due to the sheer quantity, even when attempting to download a small subsection of 500 videos. 

In [ ]:
def parse_json_field(entry):
    if entry is None:
        return []
    try:
        return json.loads(entry)
    except:
        return []

The def parse_json_field(entry) function creates a list that takes in each entry and places it into a Python list instead of the JSON strings. I used a series of statements to surf through the shofo-tiktok dataset, with the first part of the function considering if there is no entry, there won’t be anything in the list. The second if statement sees the existing entry and converts the raw text to a list that Python can understand. 


In [ ]:
def has_exact_tag(tags_json, targets):
   tags = parse_json_field(tags_json)
   targets = set(t.lower().lstrip('#') for t in targets)
   return any(str(t).lower().lstrip('#') in targets for t in tags)

This function finds and cleans each hashtag, using a loop to make the text lowercase and remove the hashtag, to then compare the clean output tag to the targeted hashtags set later for those that fall under booktok and dancetok related hashtags. 

In [ ]:
## Parses the engagement_metrics field, which is a JSON string containing various metrics like likes, shares, etc.
def parse_engagement(entry):
   if entry is None:
       return {}
   if isinstance(entry, str):
       try:
           parsed = json.loads(entry)
           return parsed if isinstance(parsed, dict) else {}
       except:
           return {}
   if isinstance(entry, dict):
       return entry
   return {}

This function is similar to the rest, but looks through the engagement metrics for the shofo-tiktok dataset. If there is no entry, the function returns an empty list. If there is an entry and a string, the function attempts to convert the raw data (the JSON) into a Python object; if that fails, the list will be empty again. The last if statement is a backup to check if the data entry is already cleaned. 

In [ ]:
## columns to read from parquet file
cols = ['video_id', 'web_url', 'creator', 'description', 'hashtags',
       'comments', 'engagement_metrics', 'date_posted', 'language']


booktok_chunks = []
dancetok_chunks = []


print("scanning shofo...")

The shofo-tiktok dataset has a lot of different data collected, but I decided to select which data columns would be the most relevant to my research topic. This section establishes the columns I selected from the shofo-tiktok TikTok database and creates empty lists.

In [ ]:
for i, batch in enumerate(pf.iter_batches(batch_size=10_000, columns=cols)):
    chunk = batch.to_pandas()

    booktok_chunks.append(
        chunk[
            chunk['hashtags'].apply(
                lambda x: has_exact_tag( #change from substring to exact match
                    x,
                    ["booktok", "bookrecs", "books", "reading", "reader", "bookreview", "bookrecommendation", "currentlyreading", "bibliophile"]
                )
            )
        ]
    )

    dancetok_chunks.append(
        chunk[
            chunk['hashtags'].apply(
                lambda x: has_exact_tag(
                    x,
                    ["dance", "choreo", "dancer", "dancetok", "dancing", "choreography", "hiphop", "ballet", "contemporary", "salsa", "tango", "ballroom", "streetdance", "breakdance", "kpopdance", "latindance", "jazzdance", "tapdance", "folkdance", "traditionaldance", "dancemusic", "dancechallenge", "dancetrend", "dancemoves", "dancetutorial", "danceduets", "dancesolo", "dancecover", "danceduo", "dancerecital", "danceperformance", "danceclass", "danceworkshop", "dancefestival", "dancecompetition", "danceshowcase", "danceday", "danceweekend", "dancevibes"]
                )
            )
        ]
    )

The next two functions processed 10,000 rows of the shofo-tiktok dataset, only taking the columns (cols) that were defined before. The videos that are selected need to have the exact tag of the listed items that I placed manually. The first section shows hashtags that relate to booktok, such as bookrecs, books, reading, etc. Meanwhile, the second section follows the same concept, inputting exact tags that relate to dancetok, such as dance, choero, dancer, etc. 

Originally, I didn’t use has_exact_tag; instead, I did the word itself as “in” the hashtag, which outputted unrelated hashtags/videos such as tags relating to “manspreading” or “devinbooker” because the word was part of the text. Ultimately, the exact tag did lower the amount of videos collected, but the videos scraped were more relevant to the booktok and dancetok communities. At the end of the processing, the data is placed into a combined dataset. ChatGBT prompt: “how to make function only search for that specific hashtag” 

In [ ]:
# Parse comments and hashtags
#separate comments into individual rows, and extract text from comments
booktok['comments_parsed'] = booktok['comments'].apply(parse_json_field)
dancetok['comments_parsed'] = dancetok['comments'].apply(parse_json_field)


booktok = booktok.explode("comments_parsed")
dancetok = dancetok.explode("comments_parsed")


booktok["comment_text"] = booktok["comments_parsed"].apply(
   lambda x: x.get("text", "") if isinstance(x, dict) else ""
)


dancetok["comment_text"] = dancetok["comments_parsed"].apply(
   lambda x: x.get("text", "") if isinstance(x, dict) else ""
)

The Shofo-TikTok dataset displayed comments as one whole row entry. This function surfs through the comments, converts them from JSON to Python lists, and then separates comments into individual rows so that analyses and comparisons on the comments will be more easily identifiable. ChatGBT prompt: “How do I separate the comments into individual rows”

In [ ]:
# Parse hashtags into lists
booktok['hashtags_parsed'] = booktok['hashtags'].apply(parse_json_field)
dancetok['hashtags_parsed'] = dancetok['hashtags'].apply(parse_json_field)


# Create a caption field (using description, or empty string if description is missing)
booktok['caption'] = booktok['description'].fillna("")
dancetok['caption'] = dancetok['description'].fillna("")

These two sections parse through hashtags and captions (used to be descriptions), taking the JSON text to become Python lists.

In [ ]:
# Make dataset labels
booktok['dataset_type'] = 'scaled_booktok'
dancetok['dataset_type'] = 'scaled_dancetok'
manual = pd.read_csv("manual_dataset.csv")

Identifies the scaled booktok and dancetok videos collected from the shofo dataset (scaled_booktok & scaled_dancetok) and also the manual dataset (the first 246 entries).

In [ ]:
comment_cols = [
   "Comment 1",
   "Comment 2",
   "Comment 3",
   "Comment 4",
   "Comment 5"
]


type_cols = [
   "Comment Type",
   "Comment Type.1",
   "Comment Type.2",
   "Comment Type.3",
   "Comment Type.4"
]

Originally, my manual dataset considered each video an entry, with rows including the top 5 comments. However, after scaling the dataset by using the Shofo dataset, I decided to make each comment an entry for easier sentiment analysis and comparison. For consistency sake, I changed the structure of my manual dataset to match the scaled dataset by separating each comment and its respective comment type that I imputed during my initial dataset. 

In [ ]:
for _, row in manual.iterrows():
   base_data = row.drop(labels=comment_cols + type_cols, errors="ignore").to_dict()


   for i in range(len(comment_cols)):
       comment = row.get(comment_cols[i])
       ctype = row.get(type_cols[i])


       if pd.notna(comment) and str(comment).strip() != "":
           rows.append({
               **base_data,
               "comment_text": str(comment).strip(),
               "comment_type": "" if pd.isna(ctype) else str(ctype).strip()
           })


manual = pd.DataFrame(rows)

To readjust my spreadsheet, the following code was generated by ChatGPT to help guide me in changing the format from wide to long. ChatGBT prompt: “how to separate comment_text, comment.type 1, comment.type 2, comment.type 3, comment.type 4 to associate with each entry. For example, entry 1 with comment type, entry 2 with comment.type 2, and entry 3 with comment.type 4”. This prompt was to thoroughly explain how I wanted the dataset to be formatted.

In [ ]:
if "Tok_Hashtag" in manual.columns:
   manual["Tok_Hashtag"] = manual["Tok_Hashtag"].fillna("")
else:
   manual["Tok_Hashtag"] = ""


if "description" in manual.columns:
   manual["caption"] = manual["description"].fillna("")
else:
   manual["caption"] = ""
manual["dataset_type"] = "manual"

This section checks the Tok_Hashtag column and fills in missing values with “”. The second part of this section makes the text within the description column to fall under the caption column instead.

In [ ]:
# Put hashtags_parsed INTO existing manual column name
booktok['Tok_Hashtag'] = booktok['hashtags_parsed'].apply(
   lambda x: " | ".join(x) if isinstance(x, list) else ""
)


dancetok['Tok_Hashtag'] = dancetok['hashtags_parsed'].apply(
   lambda x: " | ".join(x) if isinstance(x, list) else ""
)


booktok["Video Link"] = booktok["web_url"]
dancetok["Video Link"] = dancetok["web_url"]

This makes hashtags under booktok and dancetok into strings. In the Shofo dataset, the hashtags were separated with " | ". If the values are lists then at becomes a string; it returns empty. The last part of this section combines the web_url from the Shofo data into the Video Link column that was made in the manual dataset. 

In [ ]:
# combine datasets into one big table, with a column to indicate which dataset each row came from
combined = pd.concat(
   [manual, booktok, dancetok],
   ignore_index=True,
   sort=False
)

This combines the manual and Shofo datasets (booktok and dancetook) together. 

In [ ]:

# sentiment analysis on comments using VADER


combined["comment_text"] = combined["comment_text"].fillna("")


sia = SentimentIntensityAnalyzer()


combined["sentiment_score"] = combined["comment_text"].apply(
   lambda x: sia.polarity_scores(str(x))["compound"]
)


def label_sentiment(score):
   if score >= 0.05:
       return "positive"
   elif score <= -0.05:
       return "negative"
   else:
       return "neutral"


combined["sentiment"] = combined["sentiment_score"].apply(label_sentiment)

This is the sentiment analysis on the comments, using VANDER to categorize comments into one of the three categories: positive, negative, and neutral. ChatGBT prompt: “How to use VADER sentiment analysis” 

In [ ]:
# create clean sequential ID column (ONLY ONCE)
combined = combined.reset_index(drop=True)
combined.insert(0, "Sheet1l1", combined.index + 1)


cols = ["dataset_type"] + [c for c in combined.columns if c != "dataset_type"]
combined = combined[cols]

combined = combined.drop(columns=["video_id"], errors="ignore")
combined = combined.drop(columns=["web_url"], errors="ignore")


combined.to_csv("combined_dataset.csv", index=False)


print("\nCombined dataset saved!")
print(combined.head())

This last section organizes the final dataset. I created a new column to number the entries to make references to each comment more accessible. Additionally, I dropped the extra Sheet1l1 that came from my manual dataset to avoid duplication. Other final edits include placing the web_url contents into one column under Video Link.

In [ ]:
# sentiments value counts for each dataset
booktok_sentiments = booktok["sentiment"].value_counts()
dancetok_sentiments = dancetok["sentiment"].value_counts()


# print results
print("\nBookTok")
print(booktok_sentiments)


print("\nDanceTok")
print(dancetok_sentiments)


# if entry is missing, default to 0
book_pos = booktok_sentiments.get("positive", 0)
book_neg = booktok_sentiments.get("negative", 0)
book_neu = booktok_sentiments.get("neutral", 0)


dance_pos = dancetok_sentiments.get("positive", 0)
dance_neg = dancetok_sentiments.get("negative", 0)
dance_neu = dancetok_sentiments.get("neutral", 0)

This additional section helps analyze sentiment results and how BookTok and DanceTok compare with VADER Lexicon analysis on comments being either neutral, negative, or positive. If there is a null or unavailable entry, the output will default to 0, not counting the sentiment score. ChatGBT prompt: “How to output the results of sentiment analysis for DanceTok and BookTok” 

**AI Usage Disclaimer:** 
This project, under the permission of Professor Zoe and guidance of the AI Policy listed on the course website, I used ChatGPT to help debug and curate sections of my code, most notably for analyzing sentiment analysis, importing VADER, debugging function logic, and reformatting the CSV file. Ultimately, all data collection, research decisions, and writing were completed by me (the author), and all code was reviewed and cleaned if necessary. 